# Prospective concept audit, Scope B (Roadmap Work 7)

This notebook asks whether the graded concept trees, fitted on the final-layer SAE concepts of a prospective target, keep their level and ranking in later input years. It follows the locked protocol v2 (`graded-tree-protocol.md`, sections 2 and 3).

- **Target:** death in the calendar year after the input year, from the full raw DEATH events. A patient who died in the input year or earlier is removed.
- **Model inputs:** the last three calendar years (`last_3`). Reference years are 2009, 2010 and 2011, so each reference year has the full three-year window and two to four later input years.
- **Concepts:** three SAE seeds (42, 135, 215) on the reference-year embeddings. Candidates are the medoids of full factor families (cap 10%, 29 candidates). Each candidate is a ReLU activation divided by its own frozen mean of positive activations on the SAE training rows.
- **Controls:** sparsity-matched biased random directions, unbiased random directions, principal components with both signs, and a tree on record-duration features. Each control set has the same candidate count.
- **Predictor device:** the DR-TabPFN fit, the embeddings, the head gradients and the parity and finite-difference checks run on CPU, with 32 threads. Multi-year contexts do not fit the 8 GB attention of the RTX 2080, the same as the same-year builds. The SAE fit and encoding run on the GPU.
- **Runs on:** Fiji, in `~/SAE-TCAV`. This notebook is executed in place only after the user approves the run. Outputs go to `stats/prospective_target_audit/scope_b/`.
- **Scaling sensitivity:** the cached (historical) scaling sensitivity is reported in Scope A only, per plan Decision 6.

In [1]:
import json
import os
import subprocess
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import prospective_audit as audit
import prospective_graded_trees as gt

STARTED_UTC = datetime.now(timezone.utc).isoformat()
RUN_START = time.perf_counter()

REFERENCE_YEARS = (2009, 2010, 2011)
VARIANTS = ("prospective", "same_year_full", "same_year_survivor")
LAST_EVALUATION_YEAR = audit.LAST_OUTCOME_INPUT_YEAR
PREDICTOR_DEVICE = "cpu"
PREDICTOR_THREADS = 32
SAE_DEVICE = "cuda"
if SAE_DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("SAE_DEVICE is cuda but CUDA is unavailable")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
WORKERS = os.cpu_count()
BINARY_WORKERS = 24

POPULATION_PATH = Path("stats/temporal_robustness/_population/prepared.pkl")
POPULATION_SHA256 = "c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7"
EVENTS_PATH = Path("tidy_event_data.feather")
BUILDS = Path.home() / "rpi-runs/SAE-TCAV/pilot-ideas-followups/builds"
BRIDGE_CACHE_DIRECTORY = {
    2009: BUILDS / "all_history/unit-cache/128647232c48f725e8ff89ab7b954c3bfa46772485d948d8602709373e80c402",
    2010: BUILDS / "last_3/unit-cache/7fa1706899a8d5de2f76875e345d913e87f4a0f191f57c49886d3ae5161a076f",
    2011: BUILDS / "last_3/unit-cache/7fa1706899a8d5de2f76875e345d913e87f4a0f191f57c49886d3ae5161a076f",
}
OUT_DIR = Path("stats/prospective_target_audit/scope_b")
FIGURE_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FEATURES = 500
BOOTSTRAP_DRAWS = audit.BOOTSTRAP_DRAWS
BOOTSTRAP_SEED = audit.BOOTSTRAP_SEED
STRUCTURAL_DRAWS = 100
BATCH_ROWS = 256
GRADIENT_BATCH_ROWS = 512
PARITY_ROWS = 32
FINITE_DIFFERENCE_ROWS = 16
PARITY_TOLERANCE = 1e-4
PROBABILITY_ATOL = 1e-5
PROBABILITY_RTOL = 1e-4
IDENTITY_TOLERANCE = 1e-5
FINITE_DIFFERENCE_LABEL = "sus_last_layer_fd"
ARM_LABELS = ("main", "seed_42", "seed_135", "seed_215", "random_biased", "random_unbiased", "pca", "duration")

checks = []


def record_check(check, measured, rule, passed, reference_year="all", variant="all"):
    checks.append({"check": check, "variant": variant, "reference_year": str(reference_year),
                   "measured": measured, "rule": rule, "passed": bool(passed)})


print("predictor:", PREDICTOR_DEVICE, "threads", PREDICTOR_THREADS, "| SAE:", SAE_DEVICE, GPU_NAME,
      "| workers:", WORKERS, "| output:", OUT_DIR)

predictor: cpu threads 32 | SAE: cuda NVIDIA GeForce RTX 2080 | workers: 64 | output: stats/prospective_target_audit/scope_b


## Load the population and the prospective cohort

The population file is checked against its SHA-256 before it is read. Full-history raw DEATH events give each patient's first death year. A row is kept when the patient is alive at 31 December of the input year, and the outcome is the death in the next calendar year. The cohort checks confirm that no row is taken after death, that input year 2015 is dropped, and that the outcome is defined only for outcome-valid years.

The bootstrap universe is every patient in the population file. Every row inherits its patient's draw, so all years, arms and the same-year bridge are resampled on the same patients.

In [2]:
population = audit.load_population(POPULATION_PATH, POPULATION_SHA256)
events = pd.read_feather(EVENTS_PATH)
X_population = population["X"]
feature_names = list(population["feature_names"])
first_death_raw = audit.first_death_year_by_row(population, events)
cohort = audit.landmark_cohort(population, first_death_raw)
cohort_checks = audit.cohort_check_rows(cohort)
bootstrap_universe = np.unique(np.asarray(population["patient_id"], dtype=str))

record_check("population file sha256 equals the approved value", population["fingerprint"]["file_sha256"],
             POPULATION_SHA256, population["fingerprint"]["file_sha256"] == POPULATION_SHA256)
record_check("population feature count", X_population.shape[1], f"= {EXPECTED_FEATURES}",
             X_population.shape[1] == EXPECTED_FEATURES)
for check_row in cohort_checks.to_dict(orient="records"):
    record_check(check_row["check"], check_row["value"], check_row["rule"], check_row["passed"])
print("population rows:", population["fingerprint"]["rows"], "| patients:", len(bootstrap_universe),
      "| cohort rows:", len(cohort))

population rows: 21900 | patients: 5636 | cohort rows: 17547


## Prospective DR-TabPFN on CPU: embeddings, head gradients and checks

For each reference year r, the steps are:

1. **Roles and context:** the frozen patient split for year r and the last_3 context max(2007, r-2) to r. Context rows of the reference year carry next-year labels, so this model's label horizon ends on 31 December of r+1, as in Scope A.
2. **Fit and capture:** the consistent-scaling DR-TabPFN is fitted on the context rows on CPU. Embeddings (layer 11) are captured for the year-r rows of the SAE, rule, and probe roles, and for the evaluation rows of input years r to 2013. Held-at-r embeddings are captured for the same evaluation rows with the year input set to r, and each row keeps its own features X_{i,t}. For evaluation rows of year r the actual and held captures are the same call, and the checks confirm they match.
3. **Gradients and checks:** exact head gradients on the evaluation rows, a finite-difference check on the largest rows, a parity check against matched joint execution, and a replay check. Results go into `checks.csv`.

In [3]:
torch.set_num_threads(PREDICTOR_THREADS)
prospective = {}
for reference in REFERENCE_YEARS:
    cohort_r, roles = audit.add_roles(cohort, population, reference)
    row = cohort_r["row"].to_numpy()
    year = cohort_r["year"].to_numpy()
    role = cohort_r["role"].to_numpy()
    context = audit.context_mask(cohort_r, reference)
    evaluation = audit.evaluation_mask(cohort_r, reference)
    role_rows = (year == reference) & np.isin(role, ["sae_discovery", "rule_discovery", "cav_fit"])
    labels = cohort_r["y_next"].to_numpy(dtype=float)[context]

    fitted = audit.fit_dr_tabpfn_context(
        X_population[row[context]], labels, year[context], domain_origin=audit.context_first_year(reference),
        device=PREDICTOR_DEVICE, seed=audit.SEED, use_context_scaler=True)
    captured_roles = audit.capture_scored_rows(fitted, X_population[row[role_rows]], year[role_rows],
                                               batch_size=BATCH_ROWS)
    captured_eval = audit.capture_scored_rows(fitted, X_population[row[evaluation]], year[evaluation],
                                              batch_size=BATCH_ROWS)
    held_year = np.full(int(evaluation.sum()), reference)
    captured_held = audit.capture_scored_rows(fitted, X_population[row[evaluation]], held_year, batch_size=BATCH_ROWS)
    same_year = year[evaluation] == reference
    embedding_gap = np.abs(captured_eval["embeddings"][same_year] - captured_held["embeddings"][same_year]).max()
    probability_gap = np.abs(captured_eval["probabilities"][same_year]
                             - captured_held["probabilities"][same_year]).max()
    identity_gap = float(max(embedding_gap, probability_gap))
    record_check("held-at-reference identity on reference-year evaluation rows (max abs diff)", identity_gap,
                 f"<= {IDENTITY_TOLERANCE:g}", identity_gap <= IDENTITY_TOLERANCE, reference)
    gradients, fd_report = audit.gradients_with_fd(captured_eval, grad_batch_size=GRADIENT_BATCH_ROWS,
                                                   fd_rows=FINITE_DIFFERENCE_ROWS)
    parity = audit.parity_check(fitted, X_population[row[evaluation][:PARITY_ROWS]],
                                year[evaluation][:PARITY_ROWS])

    record_check("context rows used without silent subsampling", fitted["subsample_rows"],
                 f"= {fitted['context_rows']}", fitted["subsample_rows"] == fitted["context_rows"], reference)
    record_check("matched joint execution parity (max abs diff, probability or embedding)",
                 max(parity["proba_max_abs_diff"], parity["emb_max_abs_diff"]),
                 f"<= {PARITY_TOLERANCE:g}", parity["passed"], reference)
    replay_limit = PROBABILITY_ATOL + PROBABILITY_RTOL * float(np.max(captured_eval["probabilities"]))
    record_check("capture replay error (max abs)", captured_eval["max_replay_error"],
                 f"<= {replay_limit:.2e} (atol 1e-5 + rtol 1e-4 x max probability)",
                 captured_eval["max_replay_error"] <= replay_limit, reference)
    record_check(FINITE_DIFFERENCE_LABEL + " (max abs error of last-layer gradient)",
                 fd_report["max_absolute_error"], "elementwise rtol 1e-3, atol 1e-5", fd_report["passed"], reference)

    rows = np.concatenate([row[role_rows], row[evaluation]])
    prospective[reference] = {
        "rows": rows,
        "Z": np.concatenate([captured_roles["embeddings"], captured_eval["embeddings"]]).astype(np.float32),
        "Z_held": np.concatenate([captured_roles["embeddings"], captured_held["embeddings"]]).astype(np.float32),
        "gradients": np.asarray(gradients, dtype=np.float64),
        "role": np.asarray(roles["row_role"])[rows],
        "evaluation_rows": int(evaluation.sum()),
        "context_rows": int(context.sum()),
    }
    print(reference, "context:", int(context.sum()), "| evaluation rows:", int(evaluation.sum()),
          "| parity:", parity["passed"], "| FD:", fd_report["passed"])
    del fitted, captured_roles, captured_eval
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tabpfn/scripts/estimator/base.py:217: UserWarning: You are using a CPU device, but CUDA is available. This will likely lead to inferior performance.Fix this by setting device='cuda'.
  warnings.warn(


Initialized decoder for standard with (None, 10)  and nout 10


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{"absolute_error": [2.436339855194092e-06, 7.5660645961761475e-06, 3.7103891372680664e-06, 2.3618340492248535e-06, 1.55717134475708e-06, 4.06801700592041e-06, 2.942979335784912e-07, 2.2687017917633057e-06, 3.606081008911133e-06, 2.1085143089294434e-06, 4.4405460357666016e-06, 1.7881393432617188e-07, 6.332993507385254e-08, 4.719942808151245e-06, 1.4901161193847656e-06, 3.0919909477233887e-06], "analytical_derivatives": [0.05787888914346695, 0.059308972209692, 0.05411358177661896, 0.056733809411525726, 0.05664534121751785, 0.0531599223613739, 0.05274758115410805, 0.05373511090874672, 0.055058553814888, 0.052831195294857025, 0.050140514969825745, 0.05617569386959076, 0.04942261800169945, 0.050477560609579086, 0.05371123552322388, 0.05088288336992264], "atol": 1e-05, "central_derivatives": [0.057881325483322144, 0.059301406145095825, 0.054117292165756226, 0.05673617124557495, 0.056643784046173096, 0.05316399037837982, 0.052747875452041626, 0.053732842206954956, 0.05505494773387909, 0.05282

2009 context: 5023 | evaluation rows: 1408 | parity: True | FD: True


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tabpfn/scripts/estimator/base.py:217: UserWarning: You are using a CPU device, but CUDA is available. This will likely lead to inferior performance.Fix this by setting device='cuda'.
  warnings.warn(


{"absolute_error": [6.631016731262207e-07, 3.5241246223449707e-06, 5.587935447692871e-07, 1.1175870895385742e-06, 3.4943222999572754e-06, 2.7194619178771973e-07, 2.302229404449463e-06, 4.015862941741943e-06, 2.9318034648895264e-06, 9.685754776000977e-08, 6.109476089477539e-07, 5.140900611877441e-07, 4.503875970840454e-06, 2.1010637283325195e-06, 2.5816261768341064e-06, 2.2836029529571533e-06], "analytical_derivatives": [0.057032786309719086, 0.05799511820077896, 0.05341041833162308, 0.05614347755908966, 0.055888526141643524, 0.05280104652047157, 0.05249834805727005, 0.05437777191400528, 0.05493159219622612, 0.051376499235630035, 0.048959389328956604, 0.053863830864429474, 0.04624832794070244, 0.04748837649822235, 0.050445619970560074, 0.04825885221362114], "atol": 1e-05, "central_derivatives": [0.05703344941139221, 0.057991594076156616, 0.05341097712516785, 0.05614235997200012, 0.05589202046394348, 0.052800774574279785, 0.0524960458278656, 0.05438178777694702, 0.05492866039276123, 0.05

2010 context: 5563 | evaluation rows: 1138 | parity: True | FD: True


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tabpfn/scripts/estimator/base.py:217: UserWarning: You are using a CPU device, but CUDA is available. This will likely lead to inferior performance.Fix this by setting device='cuda'.
  warnings.warn(


{"absolute_error": [9.499490261077881e-06, 2.518296241760254e-06, 3.6731362342834473e-06, 4.351139068603516e-06, 5.2228569984436035e-06, 6.6086649894714355e-06, 7.450580596923828e-09, 4.511326551437378e-06, 1.4230608940124512e-06, 1.5050172805786133e-06, 5.327165126800537e-06, 3.885477781295776e-06, 3.0249357223510742e-06, 2.332031726837158e-06, 3.3564865589141846e-06, 3.416091203689575e-06], "analytical_derivatives": [0.049164704978466034, 0.04907296597957611, 0.0456080362200737, 0.04801589250564575, 0.04790649563074112, 0.045035310089588165, 0.04510582238435745, 0.046402547508478165, 0.046928904950618744, 0.04507078230381012, 0.04292156547307968, 0.047699522227048874, 0.042405396699905396, 0.04290667921304703, 0.045536454766988754, 0.04407570883631706], "atol": 1e-05, "central_derivatives": [0.049155205488204956, 0.04907548427581787, 0.04561170935630798, 0.04801154136657715, 0.047901272773742676, 0.045028701424598694, 0.045105814933776855, 0.0464070588350296, 0.04692748188972473, 0.0

2011 context: 5651 | evaluation rows: 863 | parity: True | FD: True


## Reference SAEs on the GPU and the prospective table

Three SAEs are trained for each reference year on the year-r `sae_discovery` embeddings, one per seed (42, 135, 215), with the same settings as the saved same-year studies. Every needed row is then encoded with each seed's frozen scaler and SAE. The GPU is used here since the SAE is small.

In [4]:
def make_table(rows, role, Z, codes, dictionaries, gradients):
    """One analysis table: row identities, features, duration features, embeddings, SAE codes and dictionaries."""
    patient = np.asarray(population["patient_id"], dtype=str)[rows]
    year = np.asarray(population["year"], dtype=np.int64)[rows]
    timing = audit.timing_features(pd.DataFrame({"patient_id": patient, "year": year}), events)
    return {"rows": rows, "role": np.asarray(role), "patient_id": patient, "year": year,
            "X": X_population[rows], "D": gt.duration_features(timing, X_population[rows]), "Z": Z,
            "codes": codes, "dictionaries": dictionaries, "gradients": gradients}


def subset_table(table, mask):
    keep = np.asarray(mask, dtype=bool)
    return {"rows": table["rows"][keep], "role": table["role"][keep], "patient_id": table["patient_id"][keep],
            "year": table["year"][keep], "X": table["X"][keep], "D": table["D"][keep], "Z": table["Z"][keep],
            "codes": {seed: values[keep] for seed, values in table["codes"].items()},
            "dictionaries": table["dictionaries"], "gradients": None}


tables = {"prospective": {}, "same_year_full": {}, "same_year_survivor": {}}
for reference in REFERENCE_YEARS:
    item = prospective[reference]
    year = np.asarray(population["year"], dtype=np.int64)[item["rows"]]
    sae_rows = (year == reference) & (item["role"] == "sae_discovery")
    dictionaries = gt.fit_reference_saes(item["Z"][sae_rows], SAE_DEVICE)
    codes = gt.encode_reference_saes(dictionaries, item["Z"], SAE_DEVICE)
    table = make_table(item["rows"], item["role"], item["Z"], codes, dictionaries, item["gradients"])
    table["held"] = {"Z": item["Z_held"], "codes": gt.encode_reference_saes(dictionaries, item["Z_held"], SAE_DEVICE)}
    tables["prospective"][reference] = table
    print(reference, "SAE rows:", int(sae_rows.sum()), "| latents:", next(iter(codes.values())).shape[1])
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

2009 SAE rows: 274 | latents: 288


2010 SAE rows: 291 | latents: 288


2011 SAE rows: 278 | latents: 288


## Same-year bridge: saved same-year embeddings and SAEs (read-only)

The same-year study already holds embeddings for every row from year r onward (`P/r`) and its frozen SAEs (`S/seed/r`). They are read from the saved caches and never written. r = 2009 comes from the `all_history` cache. The `last_3` cache has no 2009 unit. For 2009 the window is max(2007, 2009-2) = 2007 to 2009 in both caches, with the same domain origin, the same context rows and the same SAE configuration (seeds, epochs, alpha, learning rate, scaling factor, candidate cap, split seed). The loader checks the reference year and the roles identity of each saved output.

Held at r is not computed for the bridge: the saved caches hold embeddings only, with no fitted predictor to capture again with the year input set to r. Two cohorts are evaluated on the same protocol:

- **Full same-year cohort:** every saved row from year r onward, as the same-year study holds it.
- **Survivor cohort:** the same rows restricted to the prospective cohort, row by row. A row is kept only when its patient is alive on 31 December of that row's year (`survivor_row_mask`). This separates the change in cohort from the change in target and model.

In [5]:
bridge_caches = {}
for reference in REFERENCE_YEARS:
    directory = BRIDGE_CACHE_DIRECTORY[reference]
    if directory not in bridge_caches:
        bridge_caches[directory] = gt.open_bridge_cache(directory)
    cache = bridge_caches[directory]
    predictor = cache.load(f"P/{reference}")
    _, roles = audit.add_roles(cohort, population, reference)
    roles_match = predictor["roles_identity"] == roles["identity"]
    reference_match = int(predictor["reference_year"]) == reference
    record_check("same-year predictor roles and reference year match this run", roles_match and reference_match,
                 "saved roles identity equals recomputed roles identity, and saved reference year equals r",
                 roles_match and reference_match, reference, "same_year_full")
    rows, Z = gt.checked_bridge_embeddings(predictor, roles, population["year"], reference)
    dictionaries = gt.load_bridge_dictionaries(cache, reference, SAE_DEVICE)
    codes = gt.encode_reference_saes(dictionaries, Z, SAE_DEVICE)
    role = np.asarray(roles["row_role"])[rows]
    full = make_table(rows, role, Z, codes, dictionaries, None)
    survivor_mask = gt.survivor_row_mask(full["patient_id"], full["year"], cohort["patient_id"], cohort["year"])
    survivors = subset_table(full, survivor_mask)
    tables["same_year_full"][reference] = full
    tables["same_year_survivor"][reference] = survivors

    cohort_from_reference = int((cohort["year"].to_numpy() >= reference).sum())
    record_check("survivor rows equal the prospective cohort rows from year r on", int(survivor_mask.sum()),
                 f"= {cohort_from_reference}", int(survivor_mask.sum()) == cohort_from_reference,
                 reference, "same_year_survivor")
    raw_death_by_row = np.asarray(first_death_raw)[rows]
    on_or_after_death = int((raw_death_by_row <= full["year"]).sum())
    print(reference, "bridge rows:", len(rows), "| survivors:", int(survivor_mask.sum()),
          "| full rows on or after a raw death:", on_or_after_death)

2009 bridge rows: 17613 | survivors: 13642 | full rows on or after a raw death: 1830


2010 bridge rows: 15128 | survivors: 11415 | full rows on or after a raw death: 1572


2011 bridge rows: 12534 | survivors: 9090 | full rows on or after a raw death: 1303


## Candidates, controls and arms for each variant and reference year

For each variant (prospective, same-year full, same-year survivor) and reference year:

1. **Reference masks:** the SAE rows, the rule-discovery rows (tree fit), the cav-fit rows (probe fit), the family rows (rule plus cav), and the held-out reference evaluation rows. The evaluation rows run from year r to 2013. Held rows are the same evaluation rows with the year input set to r (prospective cohort only).
2. **Families and candidates:** a factor is dropped only if it is never positive on the SAE rows. Full families are ranked by the positive count of their medoid, and the cap is 10% of the latent count. Each candidate is the medoid column.
3. **Seed members:** the same family's member factor in each seed. Their spread across seeds is the seed-to-seed floor, and the arms are never compared by factor index.
4. **Controls:** the sparsity-matched biased random directions (bias chosen on the SAE rows so the active share equals the candidate's), unbiased random directions and principal components (both signs). Each control is normalized by its own mean of positive activations on the SAE rows.
5. **Arms:** main (clinical features X), seed members (X), three controls (X), and the record-duration tree (duration features D fitted to the main candidates).

Each candidate is one job. Its tree is fitted on the rule-discovery rows, pruned by patient-grouped cross-validation, and scored on every later input year, actual and held at r, with 500 paired patient draws. Held at r keeps every evaluation row's own clinical features X, so the tree output and leaf are the same as in the actual arm, and only the activation y differs. That activation is the embedding captured with the year input set to r, encoded with the same frozen SAE, and divided by the same frozen scale and the same control directions and biases as the actual activation.

In [6]:
def scaled_pair(columns, held_columns, sae_fit):
    """Divide the actual columns by their frozen scale (own mean of positive activations on the SAE rows); held columns,
    when given, are divided by that same scale."""
    scales = gt.factor_scale_table(columns[sae_fit])["mu_plus"].to_numpy(dtype=float)
    held = None if held_columns is None else gt.normalize_activations(held_columns, scales)
    return gt.normalize_activations(columns, scales), held


def probe_inputs(table, masks, candidates, main_normalized, evaluation, reference_rows, index):
    seed, factor = candidates[index]["medoid"]
    probe_fit = masks["probe_fit"]
    return {"Z_fit": table["Z"][probe_fit], "X_fit": table["X"][probe_fit],
            "y_fit": main_normalized[probe_fit, index], "Z_eval": table["Z"][evaluation],
            "Z_reference": table["Z"][reference_rows], "gradients": table["gradients"],
            "direction": gt.sae_factor_raw_direction(table["dictionaries"][seed], factor)}


def prepare_arms(variant, table, reference):
    year, role, patient = table["year"], table["role"], table["patient_id"]
    X, D, Z, codes = table["X"], table["D"], table["Z"], table["codes"]
    held = table.get("held")
    masks = gt.role_masks(role, year, reference)
    evaluation = gt.evaluation_year_mask(role, year, reference)
    reference_rows = np.flatnonzero(masks["reference_eval"])
    live_by_seed = {seed: gt.live_factor_mask(values[masks["sae_fit"]]) for seed, values in codes.items()}
    family = gt.family_candidate_set({seed: values[masks["family_fit"]] for seed, values in codes.items()},
                                     live_by_seed, f"{variant}/{reference}", SAE_DEVICE)
    candidates = family["selection"]["candidates"]
    main_raw = gt.candidate_columns(codes, candidates)
    members_raw = gt.seed_member_columns(codes, family["families"], candidates)
    scaler = table["dictionaries"][min(codes)]["scaler"]
    Z_standard = scaler.transform(np.asarray(Z, dtype=np.float64))
    shares = gt.factor_scale_table(main_raw[masks["sae_fit"]])["active_share"].to_numpy(dtype=float)

    # Held controls: the actual and held rows are stacked, and the directions, biases and PCA encoder are fitted on the
    # actual SAE rows only, so both blocks use the same frozen controls.
    rows_total = len(Z_standard)
    if held is None:
        stacked, fit_rows = Z_standard, masks["sae_fit"]
    else:
        record_check("held embeddings align with table rows", len(held["Z"]), f"= {rows_total}",
                     len(held["Z"]) == rows_total, reference, variant)
        stacked = np.vstack([Z_standard, scaler.transform(np.asarray(held["Z"], dtype=np.float64))])
        fit_rows = np.concatenate([masks["sae_fit"], np.zeros(rows_total, dtype=bool)])
    controls_all, _ = gt.control_factor_sets(stacked, fit_rows, shares)
    controls_raw = {name: columns[:rows_total] for name, columns in controls_all.items()}
    controls_held = None if held is None else {name: columns[rows_total:] for name, columns in controls_all.items()}
    main_raw_held = None if held is None else gt.candidate_columns(held["codes"], candidates)
    members_held = {} if held is None else gt.seed_member_columns(held["codes"], family["families"], candidates)

    main_normalized, main_held_normalized = scaled_pair(main_raw, main_raw_held, masks["sae_fit"])
    arms = {"main": (X, feature_names, main_normalized, main_held_normalized),
            "duration": (D, list(gt.DURATION_FEATURES), main_normalized, main_held_normalized)}
    for seed, columns in members_raw.items():
        arms[f"seed_{seed}"] = (X, feature_names, *scaled_pair(columns, members_held.get(seed), masks["sae_fit"]))
    for name, columns in controls_raw.items():
        held_columns = None if controls_held is None else controls_held[name]
        arms[name] = (X, feature_names, *scaled_pair(columns, held_columns, masks["sae_fit"]))

    jobs = []
    for arm, (features, names, targets, held_targets) in arms.items():
        fit_X = features[masks["tree_fit"]]
        fit_y_rows = masks["tree_fit"]
        reference_X = features[reference_rows]
        evaluation_X = features[evaluation]
        for index, record in enumerate(candidates):
            probe = None
            if variant == "prospective" and arm == "main":
                probe = probe_inputs(table, masks, candidates, main_normalized, evaluation, reference_rows, index)
            jobs.append({
                "variant": variant, "reference_year": reference, "arm": arm, "candidate": record["family_id"],
                "feature_names": list(names),
                "fit_X": fit_X, "fit_y": targets[fit_y_rows, index], "fit_patients": patient[fit_y_rows],
                "reference_X": reference_X, "reference_y": targets[reference_rows, index],
                "reference_patients": patient[reference_rows],
                "evaluation_X": evaluation_X, "evaluation_y": targets[evaluation, index],
                "held_y": None if held_targets is None else held_targets[evaluation, index],
                "evaluation_year": year[evaluation], "evaluation_patients": patient[evaluation],
                "universe": bootstrap_universe,
                "draws": BOOTSTRAP_DRAWS, "bootstrap_seed": BOOTSTRAP_SEED,
                "structural_draws": STRUCTURAL_DRAWS, "probe": probe})

    cap = family["selection"]["cap"]
    full_families = family["selection"]["full_families"]
    record_check("candidate count is min(cap, full families); cap = 10% of latents", len(candidates),
                 f"= min({cap}, {full_families})", len(candidates) == min(cap, full_families), reference, variant)
    record_check("tree-fit rows and probe rows are disjoint", int((masks["tree_fit"] & masks["probe_fit"]).sum()),
                 "= 0", not (masks["tree_fit"] & masks["probe_fit"]).any(), reference, variant)
    record_check("reference-year evaluation rows", int(len(reference_rows)), "> 0", len(reference_rows) > 0,
                 reference, variant)
    if variant == "prospective":
        record_check("gradient rows match evaluation rows", len(table["gradients"]), f"= {int(evaluation.sum())}",
                     len(table["gradients"]) == int(evaluation.sum()), reference, variant)
    sparsity = gt.sparsity_match_table(controls_raw["random_biased"], masks["sae_fit"], shares)
    sparsity = sparsity.assign(variant=variant, reference_year=reference,
                               candidate=[record["family_id"] for record in candidates])
    return {"jobs": jobs, "candidates": candidates, "main_raw": main_raw, "masks": masks,
            "sparsity_match": sparsity,
            "evaluation": evaluation, "reference_rows": reference_rows, "cap": cap,
            "family_counts": family["counts"], "live_by_seed": live_by_seed,
            "control_live": {name: int(gt.live_factor_mask(values[masks["sae_fit"]]).sum())
                             for name, values in controls_raw.items()}}


prepared = {}
for variant in VARIANTS:
    for reference in REFERENCE_YEARS:
        prepared[(variant, reference)] = prepare_arms(variant, tables[variant][reference], reference)
        print(variant, reference, "candidates:", len(prepared[(variant, reference)]["candidates"]),
              "| jobs:", len(prepared[(variant, reference)]["jobs"]),
              "| live controls:", prepared[(variant, reference)]["control_live"])
all_jobs = [job for item in prepared.values() for job in item["jobs"]]
print("total candidate jobs:", len(all_jobs))

prospective 2009 candidates: 19 | jobs: 152 | live controls: {'random_biased': 19, 'random_unbiased': 19, 'pca': 19}
prospective 2010 candidates: 16 | jobs: 128 | live controls: {'random_biased': 16, 'random_unbiased': 16, 'pca': 16}


prospective 2011 candidates: 18 | jobs: 144 | live controls: {'random_biased': 18, 'random_unbiased': 18, 'pca': 18}


same_year_full 2009 candidates: 13 | jobs: 104 | live controls: {'random_biased': 13, 'random_unbiased': 13, 'pca': 13}


same_year_full 2010 candidates: 11 | jobs: 88 | live controls: {'random_biased': 11, 'random_unbiased': 11, 'pca': 11}


same_year_full 2011 candidates: 17 | jobs: 136 | live controls: {'random_biased': 17, 'random_unbiased': 17, 'pca': 17}


same_year_survivor 2009 candidates: 13 | jobs: 104 | live controls: {'random_biased': 13, 'random_unbiased': 13, 'pca': 13}
same_year_survivor 2010 candidates: 13 | jobs: 104 | live controls: {'random_biased': 13, 'random_unbiased': 13, 'pca': 13}


same_year_survivor 2011 candidates: 18 | jobs: 144 | live controls: {'random_biased': 18, 'random_unbiased': 18, 'pca': 18}
total candidate jobs: 1104


## Run the candidate jobs and write the graded-tree tables

Each job fits one tree, runs its 100 structural bootstraps and scores all later input years in parallel processes (one process per CPU, each limited to one BLAS thread). Results are split into the prospective tables and the same-year bridge tables. The bridge tables keep the variant column, so the full and survivor cohorts can be compared with the prospective arm at the level of distributions.

Each leaf row in `graded_tree_leaves.csv` reports the year-t value with its 95% interval, and its change against the held-out reference value, paired over the same patient draws. `delta_status` is the worse of the year-t and reference statuses; "média indefinida" means the leaf (or its active rows, for the positive mean) is empty, and "sem intervalo" means it rests on exactly one row, so no interval is given (protocol §2.4).

In [7]:
results = gt.map_in_processes(gt.evaluate_candidate, all_jobs, WORKERS)
summary_all = pd.concat([item["summary"] for item in results], ignore_index=True)
year_all = pd.concat([item["year_metrics"] for item in results], ignore_index=True)
leaves_all = pd.concat([item["leaves"] for item in results], ignore_index=True)
probe_all = pd.concat([item["probe_rows"] for item in results], ignore_index=True)

is_prospective = summary_all["variant"] == "prospective"
summary_all[is_prospective].to_csv(OUT_DIR / "graded_tree_candidates.csv", index=False)
year_all[year_all["variant"] == "prospective"].to_csv(OUT_DIR / "graded_tree_year_metrics.csv", index=False)
leaves_all[leaves_all["variant"] == "prospective"].to_csv(OUT_DIR / "graded_tree_leaves.csv", index=False)
summary_all[~is_prospective].to_csv(OUT_DIR / "same_year_bridge_candidates.csv", index=False)
year_all[year_all["variant"] != "prospective"].to_csv(OUT_DIR / "same_year_bridge.csv", index=False)
leaves_all[leaves_all["variant"] != "prospective"].to_csv(OUT_DIR / "same_year_bridge_leaves.csv", index=False)

sparsity_all = pd.concat([item["sparsity_match"] for item in prepared.values()], ignore_index=True)
sparsity_all.to_csv(OUT_DIR / "control_sparsity_match.csv", index=False)
for (variant, reference), group in summary_all.groupby(["variant", "reference_year"]):
    record_check("reference mean is finite for every candidate job", int(group["reference_mean"].notna().sum()),
                 f"= {len(group)}", bool(group["reference_mean"].notna().all()), reference, variant)
print(summary_all.groupby(["variant", "reference_year", "arm"]).size().unstack("arm").to_string())
print(summary_all[summary_all["variant"] == "prospective"]["tree_status"].value_counts().to_string())

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


... (9 outputs left out)

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


/tmp/ipykernel_403440/296431475.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  probe_all = pd.concat([item["probe_rows"] for item in results], ignore_index=True)


arm                                duration  main  pca  random_biased  random_unbiased  seed_135  seed_215  seed_42
variant            reference_year                                                                                  
prospective        2009                  19    19   19             19               19        19        19       19
                   2010                  16    16   16             16               16        16        16       16
                   2011                  18    18   18             18               18        18        18       18
same_year_full     2009                  13    13   13             13               13        13        13       13
                   2010                  11    11   11             11               11        11        11       11
                   2011                  17    17   17             17               17        17        17       17
same_year_survivor 2009                  13    13   13             13   

## Linear probes and head-gradient geometry (prospective, main arm)

The probe is a RidgeCV on the 192-dimensional embeddings. It is fitted on the cav-fit rows (never the tree-fit rows) to reproduce the tree output, then scored on later years by skill, calibration and Spearman. Two sanity fits are also reported, and neither is read as a concept: a probe for the normalized activation itself, reported as R-squared on the held-out reference rows and as cosines with the SAE encoder direction, and the cosine between the tree probe and that direction. Principal-component energy shows how the probe weight is spread over the embedding's components. Gradient geometry gives the angle between the probe weight and the mean head gradient of each year's evaluated patients, and the mean directional derivative of the death probability along the probe direction. The sign count used by TCAV is not reported as a primary metric, since it saturates under a nearly one-dimensional gradient (Work 2).

In [8]:
probe_frame = probe_all[probe_all["variant"] == "prospective"]
probe_frame[probe_frame["section"].isin(["probe_metrics", "sanity", "energy"])].to_csv(
    OUT_DIR / "linear_probes.csv", index=False)
probe_frame[probe_frame["section"] == "geometry"].to_csv(OUT_DIR / "probe_gradient_geometry.csv", index=False)
print(probe_frame[(probe_frame["section"] == "sanity")].groupby(["reference_year", "metric"])["estimate"]
      .describe()[["count", "mean", "min", "max"]].to_string())

                                            count      mean       min       max
reference_year metric                                                          
2009.0         cosine_activation_direction   19.0  0.127475 -0.115570  0.309630
               cosine_tree_direction         19.0  0.064334 -0.098381  0.207450
               r2_reference_activation       19.0  0.937011  0.812830  0.992505
2010.0         cosine_activation_direction   16.0  0.167838 -0.029700  0.398677
               cosine_tree_direction         16.0  0.094061 -0.057114  0.288616
               r2_reference_activation       16.0  0.945373  0.840726  0.987956
2011.0         cosine_activation_direction   18.0  0.124130 -0.086543  0.354785
               cosine_tree_direction         18.0  0.079605 -0.153118  0.319109
               r2_reference_activation       18.0  0.917832  0.790809  0.979805


## Secondary binary-rule arm (prospective SAE candidates)

The frozen binary-rule sweep is run on the same rule-discovery rows, with the settings saved in each reference year's bridge cache (`threshold_sweep`, `rule`, `split_seed`). Each frozen rule R is applied to later years, and A is the candidate's activation above its frozen cut-off. The table compares P(A | R) with AUROC(activation, R) for each input year, with the same patient draws. This replicates the earlier pairing of a rule's conditional share with its ranking, next to the graded primary arm.

In [9]:
def binary_arm_job(job):
    """One frozen rule on one reference year, with its metrics for every later input year."""
    arguments = {name: value for name, value in job.items() if name != "keys"}
    return gt.frozen_rule_year_rows(**arguments).assign(**job["keys"])


binary_jobs, rule_records, denominator_rows = [], [], []
for reference in REFERENCE_YEARS:
    config = bridge_caches[BRIDGE_CACHE_DIRECTORY[reference]].config
    table = tables["prospective"][reference]
    item = prepared[("prospective", reference)]
    tree_fit, evaluation = item["masks"]["tree_fit"], item["evaluation"]
    sweep = gt.run_binary_rule_arm(
        table["X"][tree_fit], table["patient_id"][tree_fit], feature_names, item["candidates"],
        item["main_raw"][tree_fit], config["threshold_sweep"], config["rule"], reference_year=reference,
        split_seed=config["split_seed"], workers=BINARY_WORKERS, rule_source=config["rule"]["rule_source"],
        recurrence=config["rule"]["recurrence"])
    candidate_ids = [record["family_id"] for record in item["candidates"]]
    for entry in sweep["frozen"]:
        column = candidate_ids.index(entry["family_id"])
        binary_jobs.append({
            "entry": entry, "X": table["X"][evaluation], "activation": item["main_raw"][evaluation, column],
            "years": table["year"][evaluation], "patients": table["patient_id"][evaluation],
            "universe": bootstrap_universe, "draws": BOOTSTRAP_DRAWS, "seed": BOOTSTRAP_SEED,
            "keys": {"reference_year": reference, "family_id": entry["family_id"]}})
        rule_records.append({"reference_year": reference, "family_id": entry["family_id"],
                             "activation_target": entry["activation_target"], "cutoff": entry["cutoff"],
                             "rule_count": entry["rule_count"], "condition_count": entry["condition_count"],
                             "rule_identity": entry["rule_identity"], "tune_f2": entry["tune_metrics"]["f2"]})
    without_frozen = sum(family["status"] != "frozen" for family in sweep["families"])
    denominator_rows.append({"variant": "prospective", "reference_year": reference, "candidates": len(candidate_ids),
                             "frozen_rules": len(sweep["frozen"]), "without_frozen_rule": without_frozen})
    print(reference, "candidates:", len(candidate_ids), "| frozen rules:", len(sweep["frozen"]),
          "| without frozen rule:", without_frozen)

binary_results = gt.map_in_processes(binary_arm_job, binary_jobs, BINARY_WORKERS)
binary_table = pd.concat(binary_results, ignore_index=True)
binary_table.to_csv(OUT_DIR / "binary_rule_secondary.csv", index=False)
pd.DataFrame(rule_records).to_csv(OUT_DIR / "binary_rule_rules.csv", index=False)
pd.DataFrame(denominator_rows).to_csv(OUT_DIR / "binary_rule_denominator.csv", index=False)

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


2009 candidates: 19 | frozen rules: 11 | without frozen rule: 8


2010 candidates: 16 | frozen rules: 14 | without frozen rule: 2


2011 candidates: 18 | frozen rules: 15 | without frozen rule: 3


## Figures, checks table and run record

The first figure shows the median across candidates of the skill against the frozen reference mean, for each arm, by input year (prospective, actual rows). The second shows the median hurdle AUROC of the main arm for the prospective cohort and the two same-year cohorts. Medians are used since the candidates are different factors, so only the distributions of each arm are compared, not paired factor indices. The seed-to-seed floor is the spread across the seed-member arms, read from the same figure.

The checks table is written before the notebook decides anything. If any check failed, the notebook stops with an error after writing the table and the run record.

In [10]:
year_actual = year_all[(year_all["comparison"] == "actual") & (year_all["variant"] == "prospective")]


def plot_arm_medians(metric, path):
    fig, axes = plt.subplots(1, len(REFERENCE_YEARS), figsize=(4.4 * len(REFERENCE_YEARS), 3.6),
                             sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for axis, reference in zip(axes, REFERENCE_YEARS):
        part = year_actual[(year_actual["reference_year"] == reference) & (year_actual["metric"] == metric)]
        for arm in ARM_LABELS:
            median = part[part["arm"] == arm].groupby("input_year")["estimate"].median()
            axis.plot(median.index, median.values, marker="o", markersize=3, label=arm)
        axis.set_title(f"reference year {reference}")
        axis.set_xlabel("input year")
    axes[0].set_ylabel(f"median across candidates: {metric}")
    axes[0].legend(fontsize=7)
    fig.savefig(path, dpi=120)
    plt.close(fig)


def plot_cohort_medians(metric, path):
    fig, axes = plt.subplots(1, len(REFERENCE_YEARS), figsize=(4.4 * len(REFERENCE_YEARS), 3.6),
                             sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    main_actual = year_all[(year_all["comparison"] == "actual") & (year_all["arm"] == "main")
                           & (year_all["metric"] == metric)]
    for axis, reference in zip(axes, REFERENCE_YEARS):
        for variant in VARIANTS:
            part = main_actual[(main_actual["reference_year"] == reference) & (main_actual["variant"] == variant)]
            median = part.groupby("input_year")["estimate"].median()
            axis.plot(median.index, median.values, marker="o", markersize=3, label=variant)
        axis.set_title(f"reference year {reference}")
        axis.set_xlabel("input year")
    axes[0].set_ylabel(f"median across candidates: {metric}")
    axes[0].legend(fontsize=7)
    fig.savefig(path, dpi=120)
    plt.close(fig)


plot_arm_medians("skill", FIGURE_DIR / "skill_by_arm.png")
plot_cohort_medians("hurdle_auroc", FIGURE_DIR / "main_arm_hurdle_by_cohort.png")

checks_table = pd.DataFrame(checks)
checks_table.to_csv(OUT_DIR / "checks.csv", index=False)


def git_text(*args):
    completed = subprocess.run(["git", *args], capture_output=True, text=True, check=True)
    return completed.stdout.strip()


def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


run_record = {
    "work": "prospective-target-audit", "scope": "B", "notebook": "prospective_concept_audit.ipynb",
    "started_utc": STARTED_UTC, "finished_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds": round(time.perf_counter() - RUN_START, 1),
    "git_commit": git_text("rev-parse", "HEAD"),
    "uncommitted_entries": git_text("status", "--porcelain").splitlines(),
    "population_sha256": population["fingerprint"]["file_sha256"],
    "predictor_device": PREDICTOR_DEVICE, "predictor_threads": PREDICTOR_THREADS,
    "sae_device": SAE_DEVICE, "gpu_name": GPU_NAME, "torch_cuda_version": torch.version.cuda,
    "candidate_workers": WORKERS, "binary_arm_workers": BINARY_WORKERS,
    "worker_blas_threads": gt.WORKER_BLAS_THREADS, "reference_years": list(REFERENCE_YEARS), "variants": list(VARIANTS),
    "bootstrap": {"draws": BOOTSTRAP_DRAWS, "seed": BOOTSTRAP_SEED},
    "structural_draws": STRUCTURAL_DRAWS, "sae_seeds": list(gt.SAE_SEEDS),
    "package_versions": {name: package_version(name) for name in
                         ("torch", "numpy", "pandas", "scikit-learn", "scipy", "tabpfn", "matplotlib")},
    "bridge_cache_directories": {str(reference): str(path) for reference, path in BRIDGE_CACHE_DIRECTORY.items()},
    "candidate_jobs": len(all_jobs), "binary_rule_jobs": len(binary_jobs),
    "cohort_counts": {variant: {str(reference): {"rows": int(len(tables[variant][reference]["rows"])),
                                                 "candidates": len(prepared[(variant, reference)]["candidates"]),
                                                 "family_counts": prepared[(variant, reference)]["family_counts"],
                                                 "live_controls": prepared[(variant, reference)]["control_live"]}
                                      for reference in REFERENCE_YEARS} for variant in VARIANTS},
    "checks_passed": int(checks_table["passed"].sum()), "checks_total": int(len(checks_table)),
}
run_record["outputs"] = sorted(str(path.relative_to(OUT_DIR)) for path in OUT_DIR.rglob("*") if path.is_file())
(OUT_DIR / "run_record.json").write_text(json.dumps(run_record, indent=2, sort_keys=True, default=str))

print(checks_table.to_string(index=False))
failed = checks_table.loc[~checks_table["passed"].astype(bool), ["check", "variant", "reference_year"]]
if len(failed):
    raise RuntimeError(f"{len(failed)} check(s) failed, see checks.csv:\n{failed.to_string(index=False)}")
print(f"all {len(checks_table)} checks passed; outputs in {OUT_DIR}")

                                                                      check            variant reference_year                                                         measured                                                                                     rule  passed
                           population file sha256 equals the approved value                all            all c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7                         c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7    True
                                                   population feature count                all            all                                                              500                                                                                    = 500    True
                            rows with raw DEATH on or before the input year                all            all                                                                0          